# 03 · Model exploration

Compare indicators, look at correlations, try normalisations and experiment with preliminary scoring.

> ⚠️ **These notebooks run on the SYNTHETIC demonstration data.** Every number below is fictional and says nothing about real Catalan grid nodes. To explore real data, change `cfg.mode_paths("demo")` to `cfg.mode_paths("real")` once the real stages have run.

**All weights are illustrative.** Exploration here should *inform* `config/scoring.yaml`, not replace it.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from bess_catalonia.config import load_config
from bess_catalonia.pipeline import run_pipeline

cfg = load_config()  # finds the project root from the notebook's location
paths = cfg.mode_paths("demo")  # SYNTHETIC demonstration outputs (results/demo/)

# Make sure the demo outputs exist (fast no-op when they are up to date).
if not (paths.processed_root / "integrate" / "_stage.json").exists():
    run_pipeline("demo", run_all=True, root=cfg.root, log_level="WARNING")
print("Project root:", cfg.root)
print("Reading:", paths.results_root)

## 1. Integrated dataset and configured indicators

In [ ]:
nodes = pd.read_csv(paths.processed_root / "integrate" / "nodes_integrated.csv")
ind_cfg = cfg.scoring["indicators"]
indicators = list(ind_cfg)
pd.DataFrame(ind_cfg).T

## 2. Correlations between indicators
Strongly correlated indicators double-count the same signal if both carry weight.

In [ ]:
corr = nodes[indicators].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(indicators)), indicators, rotation=90)
ax.set_yticks(range(len(indicators)), indicators)
fig.colorbar(im, label="Pearson r"); ax.set_title("Indicator correlations (synthetic)")
plt.tight_layout()

## 3. Normalisation: min-max vs percentile

In [ ]:
from bess_catalonia.scoring.model import normalise

col = "renewable_mw_10km"
cmp = pd.DataFrame({"raw": nodes[col],
                    "minmax": normalise(nodes[col], "minmax", "higher_better"),
                    "percentile": normalise(nodes[col], "percentile", "higher_better")})
ax = cmp.plot.scatter(x="minmax", y="percentile", figsize=(4.5, 4))
ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_title(col)
cmp.describe().T

## 4. Preliminary scoring experiment
The pipeline's `score_scenario` is a pure function, so you can try alternative weights without touching the pipeline.

In [ ]:
from bess_catalonia.scoring.model import score_scenario

my_weights = {"erni_12m_gwh": 0.5, "bess_headroom_proxy_mw": 0.3, "suitable_land_ha_5km": 0.2}
mine = score_scenario(nodes, my_weights, ind_cfg, cfg.scoring["min_weight_coverage"],
                      cfg.scoring["exclude_if_true"], "my_experiment")
official = pd.read_csv(paths.processed_root / "score" / "scores_long.csv")
ref = official[official.scenario == "renewable_absorption"].set_index("node_id")["rank"]
mine.set_index("node_id").assign(renewable_absorption_rank=ref).sort_values("rank").head(10)

## 5. A first look at sensitivity (exploratory)
Randomly perturb the scenario weights by ±20 % and see how often each node stays in the top 5. The formal sensitivity analysis is deferred until real data exists (`docs/future_features.md`).

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
scen = cfg.scoring["scenarios"]["renewable_absorption"]["weights"]
top5 = pd.Series(0, index=nodes["node_id"])
n_runs = 200
for _ in range(n_runs):
    w = {k: v * rng.uniform(0.8, 1.2) for k, v in scen.items()}
    w = {k: v / sum(w.values()) for k, v in w.items()}
    r = score_scenario(nodes, w, ind_cfg, cfg.scoring["min_weight_coverage"],
                       cfg.scoring["exclude_if_true"])
    top5[r.loc[r["rank"] <= 5, "node_id"]] += 1
(top5 / n_runs).sort_values(ascending=False).head(10).to_frame("share of runs in top 5")